# K513 · Week 4 Homework
## Pricing used cars — and knowing when your model is too big for its data

This assignment covers **both** of this week's sessions. Part 1 is Tuesday: build a regression on a
table that has text columns in it, and read what it says in dollars. Part 2 is Thursday: take most of
the data away and use the **evaluation steps** to work out what to do about it.

**The dataset.** `Used_Toyota.csv` — 1,416 used Toyota Corollas offered for sale.

| Column | What it is |
|---|---|
| `Price` | asking price in dollars — **the target** |
| `Age_Month` | age of the car in months |
| `KM` | mileage |
| `Engine_Size` | engine size in litres |
| `Num_Doors` | 2/3-Doors, 4/5-Doors, or Stationwagen |
| `Mfr_Guarantee` | still inside the manufacturer's guarantee (1 = yes) |
| `Power_Steering` | has power steering (1 = yes) |

**What gets graded.** Five short written answers and one Analyst's Note. The code matters because it
produces the numbers, but the numbers are not the deliverable — what you conclude from them is.

**Where the answers go.** Not in this notebook — in the Canvas quiz **Week 4 Homework - Regression**. Work them out here,
then write them there. Nothing typed into this notebook is collected, so an answer left here scores
zero.

**Due Sunday 11:59 pm.** Submit the Canvas quiz, and paste a link to this notebook into its first
question.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

AI will also write code cells out of order and leave redundant ones behind. **Clean the notebook
before you submit it.** A notebook that only runs top to bottom if you know which cells to skip is
not a finished piece of work.

The specific trap this week is Part 3. An AI will write you a confident Analyst's Note in four
seconds, and it will not match the numbers in your own table, because it cannot see them. A note that
contradicts your own output is worse than no note at all — it is the fastest way to show you did not
read your own results.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one. If
anything ever looks wrong: **Runtime → Restart session and run all**.

---
## 0 · Setup

Run all three cells. Nothing here needs changing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option('display.precision', 3)
random_seed = 42

In [ ]:
TOYOTA_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/Used_Toyota.csv"

toyota_df = pd.read_csv(TOYOTA_URL)
toyota_df.head()

In [ ]:
toyota_df.info()

---
# Part 1 · Build the model

## 1.1 · Which columns are numbers, and which only look like numbers?

`info()` tells you what Python thinks each column is. It does not tell you what the column *means*,
and that is the decision you have to make.

A column is **continuous** if the arithmetic is meaningful — 60 months really is twice 30 months. A
column is **categorical** if the values are labels, even when they are stored as numbers.

`Engine_Size` is the one worth stopping on. It stores 1.3, 1.4, 1.6, 1.8, 1.9 and 2.0. Ask yourself
whether a 2.0 litre engine is worth exactly twice a 1.0 litre one, and whether the step from 1.3 to
1.4 is the same kind of thing as the step from 1.9 to 2.0.

In [ ]:
# Print the distinct values and their counts for each column that might be categorical.
for column in ['Engine_Size', 'Num_Doors', 'Mfr_Guarantee', 'Power_Steering']:
    print(toyota_df[column].value_counts().to_string(), '\n')

**(a)** Which columns did you treat as continuous, and which as categorical? Name `Engine_Size`
explicitly and say why you put it where you did. Two or three sentences.

> ✏️ **Answer in Canvas** — *Week 4 Homework - Regression*, question **(a)**.

## 1.2 · Look at it before you model it

Two pictures, both from Week 2.

In [ ]:
sns.pairplot(toyota_df[['Price', 'Age_Month', 'KM']])
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=toyota_df, x='Num_Doors', y='Price', ax=axes[0])
sns.boxplot(data=toyota_df, x='Engine_Size', y='Price', ax=axes[1])
plt.tight_layout()
plt.show()

## 1.3 · Set up X and y, and split

▶ [Video walkthrough — setting up X and y](https://youtu.be/k24lzwPee_s)

In [ ]:
X = toyota_df.drop(columns=['____'])
y = toyota_df['____']

continuous_features  = ['Age_Month', 'KM']
categorical_features = ['Engine_Size', 'Num_Doors', 'Mfr_Guarantee', 'Power_Steering']

print('X shape:', X.shape, '  y shape:', y.shape)

Split next, and split **before** anything else touches the data. Use 25% for the test set and
`random_state=random_seed`.

Check your numbers against these before you go on. If they do not match, the rest of your answers
will not match either.

```
training rows: 1062
test rows:      354
```

▶ [Video walkthrough — splitting into train and test](https://youtu.be/3U9bm7Dtbek)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=____, random_state=____)

print("training rows:", X_train.shape[0])
print("test rows:    ", X_test.shape[0])

## 1.4 · The baseline

Nothing can be called a good model until something else is worse. The baseline predicts the same
price for every car: the **training** average.

In [ ]:
guess = np.full(len(y_test), y_train.mean())

print(f"guess the average: ${y_train.mean():,.0f}")
print(f"  RMSE  ${np.sqrt(mean_squared_error(y_test, guess)):,.0f}")
print(f"  MAE   ${mean_absolute_error(y_test, guess):,.0f}")

## 1.5 · Preprocess, then fit

`Age_Month` and `KM` are already numbers, so nothing needs doing to them. The four categorical
columns have to become numbers before a model can multiply them by anything — that is what
`OneHotEncoder` is for, and `drop='first'` leaves one level out as the reference.

Nothing is standardized here. That keeps the coefficients in dollars, which is the point of
Part 1.

▶ [Video walkthrough — building the preprocessor](https://youtu.be/c5UB8ZNN5A8)

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', ____),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ____),
])

▶ [Video walkthrough — assembling the pipeline](https://youtu.be/RhRU5npGsLg)

In [ ]:
model_1 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])

model_1.fit(X_train, y_train)

## 1.6 · Score it

▶ [Video walkthrough — making predictions](https://youtu.be/oFmIHkdNH3Y)

▶ [Video walkthrough — scoring the model](https://youtu.be/O2Gi2Ku8i4c)

In [ ]:
predicted_train = model_1.predict(X_train)
predicted_test  = model_1.predict(X_test)

print(f"train R2  {model_1.score(X_train, y_train):.3f}")
print(f"test  R2  {model_1.score(X_test,  y_test):.3f}")
print(f"test  RMSE  ${np.sqrt(mean_squared_error(y_test, predicted_test)):,.0f}")
print(f"test  MAE   ${mean_absolute_error(y_test, predicted_test):,.0f}")

**(b)** Compare the model to the baseline. By how many dollars of RMSE does it beat guessing the
average, and what does that difference mean to somebody buying and reselling cars for a living? Two
or three sentences.

> ✏️ **Answer in Canvas** — *Week 4 Homework - Regression*, question **(b)**.

## 1.7 · What the model is saying, in dollars

▶ [Video walkthrough — reading the coefficients](https://youtu.be/Mci36QgLJP0)

In [ ]:
feature_names = model_1.named_steps['preprocessor'].get_feature_names_out()
coefficients  = model_1.named_steps['regressor'].coef_

print(f"Intercept: ${model_1.named_steps['regressor'].intercept_:,.2f}\n")
for name, value in zip(feature_names, coefficients):
    print(f"  {name:32} {value:>12,.2f}")

**(c)** Write out **two** coefficients as full sentences a non-technical person would understand —
one continuous column and one categorical column. Include the *holding everything else constant*
qualifier, and for the categorical one say what it is being compared **to**.

Then name the one coefficient in this table you trust least, and say why.

> ✏️ **Answer in Canvas** — *Week 4 Homework - Regression*, question **(c)**.

---
# Part 2 · What if you had almost no data?

Your dealership is opening in a new city. The model above was built on 1,062 sales — but in the new
city you have only sold a handful of cars so far, and the pricing decisions start on Monday.

This part is Thursday's session applied to your own model.

Run the setup cell. It gives you `fit_and_score(n_cars, model)` — the same helper shape as Thursday,
one line with two decisions in it: **how many cars you have sold**, and **which model**.

One small thing worth noticing in the code: `handle_unknown='ignore'` matters much more here than it
did in Part 1. With only a dozen cars on record you may never have sold a 1.9 litre one, so the
encoder has to be told what to do when it meets a level it has not seen.

**You will see a yellow `UserWarning` about unknown categories on the 12-car rows. That is not a bug.**
It is the encoder telling you that a car came up for pricing with an engine size you have never sold.
Read it rather than scrolling past it — it is the small-sample problem announcing itself before any
score has been printed.

In [ ]:
SEED = 15          # fixes which cars you have "sold so far", so everyone gets the same numbers


def fit_and_score(n_cars, model, label=None):
    """Fit `model` on a sample of n_cars, score it on train and on the same 354 test cars."""
    rng  = np.random.RandomState(SEED)
    rows = rng.choice(len(X_train), n_cars, replace=False)
    X_small, y_small = X_train.iloc[rows], y_train.iloc[rows]

    prep = ColumnTransformer(transformers=[
        ('num', StandardScaler(), continuous_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
    ])
    pipe = Pipeline(steps=[('preprocessor', prep), ('regressor', model)]).fit(X_small, y_small)

    predicted = pipe.predict(X_test)
    name = label or f"{type(model).__name__}, {n_cars} cars"
    print(f"{name:<30} train R2 {pipe.score(X_small, y_small):6.3f}"
          f"   test R2 {pipe.score(X_test, y_test):7.3f}"
          f"   RMSE ${np.sqrt(mean_squared_error(y_test, predicted)):>6,.0f}"
          f"   MAE ${mean_absolute_error(y_test, predicted):>6,.0f}")


def small_baseline(n_cars):
    rng  = np.random.RandomState(SEED)
    rows = rng.choice(len(X_train), n_cars, replace=False)
    guess = np.full(len(y_test), y_train.iloc[rows].mean())
    print(f"{'Guess the average of ' + str(n_cars):<30} train R2    ---   test R2     ---"
          f"   RMSE ${np.sqrt(mean_squared_error(y_test, guess)):>6,.0f}"
          f"   MAE ${mean_absolute_error(y_test, guess):>6,.0f}")

## 2.1 · Take the data away

**(d)** Fill in the three sample sizes and run it.

In [ ]:
small_baseline(30)
fit_and_score(____, LinearRegression(), label="300 cars sold")
fit_and_score(____, LinearRegression(), label="30 cars sold")
fit_and_score(____, LinearRegression(), label="12 cars sold")

**(d) continued.** Run the **evaluation steps** on the **30-car** row and then on the **12-car** row. Name the
step number you land on for each, and say what that step tells you to do.

They are not the same step. That is the whole of Part 2.

> ✏️ **Answer in Canvas** — *Week 4 Homework - Regression*, question **(d)**.

## 2.2 · Does a penalty help?

**(e)** Try three penalty strengths at **each** of the two sample sizes. Report all six rows.

In [ ]:
print("--- 12 cars sold ---")
fit_and_score(12, LinearRegression(),  label="12 cars, no penalty")
fit_and_score(12, Ridge(alpha=____),   label="12 cars, alpha 1")
fit_and_score(12, Ridge(alpha=____),   label="12 cars, alpha 5")

print("\n--- 30 cars sold ---")
fit_and_score(30, LinearRegression(),  label="30 cars, no penalty")
fit_and_score(30, Ridge(alpha=____),   label="30 cars, alpha 1")
fit_and_score(30, Ridge(alpha=____),   label="30 cars, alpha 5")

**(e) continued.** The penalty helps one of these two situations and hurts the other. Say which is
which, and explain **why** — what is different about the two situations that makes the same tool
right in one and wrong in the other? Three or four sentences.

*(This is the most important written answer in the assignment. An answer that says "regularization
reduces overfitting" has restated the definition without using either of your tables.)*

> ✏️ **Answer in Canvas** — *Week 4 Homework - Regression*, question **(e)**.

---
# Part 3 · The Analyst's Note

**About 150 words. This is worth a meaningful share of the grade.**

Your dealership's regional manager does not code and has about ninety seconds. Write her a note that
answers this:

> *"We're opening in the new city on Monday and we've only sold a dozen cars there so far. Can we use
> the pricing model from head office, or do we need to build a new one? What would you actually do?"*

A good note:

- gives her **a decision**, not a summary of what you tried
- quotes **numbers from your own tables above**, in dollars
- says what the recommendation **cannot** do — the limitation you would want on the record if it
  went wrong
- does not use the words *overfitting*, *regularization*, *R²* or *alpha*

That last constraint is not a style exercise. If you can only explain the recommendation in the
vocabulary of the course, you cannot yet explain it to the person who has to act on it.

> ✏️ **Answer in Canvas** — *Week 4 Homework - Regression*, question **Part 3 · The Analyst's Note**.

---
### Before you submit

- [ ] **Runtime → Restart session and run all.** It must run top to bottom with no errors.
- [ ] Every `____` filled in.
- [ ] All six written answers submitted in the Canvas quiz **Week 4 Homework - Regression** — 1.1(a), 1.6(b), 1.7(c),
      2.1(d), 2.2(e), and the note.
- [ ] This notebook shared — **Share → General access → Anyone with the link** — and its link pasted
      into the quiz's first question. Check it opens in a private window.
- [ ] Any cells an AI left behind, out of order or duplicated, cleaned up.
- [ ] Your name in the filename.